# **01_전이학습**

ImageNet 사전학습 ResNet18을 이용해 연예인 4명 분류 문제를 풀어봅니다.

- 폴더 구조로 이미지 데이터를 준비합니다.
- `ImageFolder`와 `DataLoader`로 학습 데이터를 만듭니다.
- 전이학습의 두 방식인 `Feature Extraction`과 `Fine-tuning`을 비교합니다.
- 처음부터 CNN을 학습하는 방식과 왜 차이가 나는지 확인합니다.

## 1.환경준비

### (0) 코랩 한글 패치

In [ ]:
# Colab 한글 폰트 설정
# 코드 실행 후 런타임 재시작
!sudo apt-get update -qq
!sudo apt-get install -y fonts-nanum
!rm -rf ~/.cache/matplotlib

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 코랩 한글 테스트
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.plot([1, 2, 3], [10, 20, 15])
plt.title("한글 제목 테스트")
plt.xlabel("가로축")
plt.ylabel("세로축")
plt.show()

### (1) 라이브러리 Import

In [ ]:
import os
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
# import matplotlib.pyplot as plt

# # plt.rcParams['font.family'] = 'Malgun Gothic'
# # plt.rcParams['axes.unicode_minus'] = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### (2) 전이학습 개념

ImageNet 같은 대규모 데이터로 미리 학습된 모델의 지식을 **새로운 문제에 재활용**하는 기법입니다.

**왜 필요한가**
- 내 데이터는 보통 수십~수천 장 수준이라 처음부터 학습하면 과적합되기 쉽습니다.
- 전이학습을 사용하면 적은 데이터로도 비교적 높은 성능을 기대할 수 있습니다.

**두 가지 방식**

| 방식 | 설명 | 언제 쓰나 |
|---|---|---|
| **Feature Extraction** | 사전학습 층을 모두 동결하고 마지막 분류층만 학습 | 데이터가 매우 적을 때 |
| **Fine-tuning** | 일부 층 또는 전체 층을 작은 학습률로 함께 재학습 | 데이터가 조금 더 있을 때 |

### (3) 대표 사전학습 모델

| 모델 | 특징 |
|---|---|
| **VGG16 / VGG19** | 단순 Conv-Pool 반복, 파라미터 많음 |
| **ResNet18 / 50** | Skip connection, 가장 자주 쓰임 |
| **EfficientNet** | 성능 대비 파라미터 효율 좋음 |
| **ViT** | Vision Transformer, 최신 트렌드 |

입문 실습에서는 작고 빠른 **ResNet18**을 사용합니다.

마지막 `fc`가 `Linear(512, 1000)`인 이유는 ImageNet 1000개 클래스를 분류하도록 학습되었기 때문입니다.

우리 문제는 4명 분류이므로 마지막 층만 `Linear(512, 4)`로 바꿉니다.

### (4) Custom Dataset 기본 구조

내가 수집한 이미지/CSV를 PyTorch에서 직접 쓰려면 `Dataset` 클래스를 상속받아 세 메서드를 구현합니다.

| 메서드 | 역할 |
|---|---|
| `__init__` | 파일 경로/라벨 등 초기 설정 |
| `__len__` | 전체 샘플 수 |
| `__getitem__(i)` | i번째 샘플을 `(X, y)`로 반환 |

이번 실습의 실제 학습은 더 간단한 `ImageFolder`를 사용합니다.

In [ ]:
# 이미지 폴더 + 라벨 csv 형태의 Custom Dataset 예시 (참고용)
import pandas as pd

class ImageDataset(Dataset):
    def __init__(self, img_dir, labels_csv, transform=None):
        self.img_dir = img_dir
        self.labels = pd.read_csv(labels_csv)   # 첫 컬럼: 파일명, 둘째: 라벨
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        img_path = os.path.join(self.img_dir, self.labels.iloc[i, 0])
        img = Image.open(img_path).convert('RGB')
        label = self.labels.iloc[i, 1]
        if self.transform:
            img = self.transform(img)
        return img, label

print('ImageDataset 클래스 정의 완료 (참고용 — 실제 학습은 ImageFolder 로)')

## 2.데이터 준비

### (1) 실습 데이터 다운로드

폴더 구조를 아래처럼 만들면 `ImageFolder`가 폴더명을 자동으로 클래스 라벨로 사용합니다.

```text
data/celeb/
  강호동/
  카리나/
  이수지/
  차은우/
```

이미 데이터가 있으면 다운로드 셀은 건너뛸 수 있습니다.

In [ ]:
# %pip install bing_image_downloader -q

In [ ]:
# 데이터 다운로드 — 이미 있으면 건너뜀
'''
from bing_image_downloader import downloader

queries = ['마동석', '카리나', '이수지', '아이유']
if not os.path.isdir('./data/celeb/마동석'):
    for q in queries:
        downloader.download(q, limit=40, output_dir='./data/celeb',
                            adult_filter_off=True, timeout=60)
else:
    print('이미 다운로드됨')

# 깨진 이미지 정리 — Bing 결과에 가끔 손상된 파일이 섞임
for cls in os.listdir('./data/celeb'):
    for f in os.listdir(f'./data/celeb/{cls}'):
        fp = f'./data/celeb/{cls}/{f}'
        try:
            with Image.open(fp) as img:
                img.verify()
        except Exception:
            os.remove(fp)
print('데이터 정리 완료')
'''

### (2) Transform + ImageFolder + DataLoader

사전학습 모델을 사용할 때는 사전학습에 사용된 ImageNet 평균/표준편차로 정규화합니다.

- 입력 크기: `224 x 224`
- 정규화 값: ImageNet `MEAN`, `STD`
- 데이터 분할: train/test = 8:2
- `stratify`로 클래스 비율 유지

In [ ]:
# ImageNet 사전학습 모델용 정규화 값 (외울 필요는 없고 그냥 가져다 쓰기)
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

dataset = datasets.ImageFolder(
    root='./data/celeb',
    transform=transform,
    loader=lambda p: Image.open(p).convert('RGB'),  # RGBA 등 안전하게 RGB 로
)

# 8:2 분할 — stratify 로 4명 비율 유지
indices = list(range(len(dataset)))
train_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=dataset.targets
)
train_set = Subset(dataset, train_idx)
test_set = Subset(dataset, test_idx)

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
test_loader = DataLoader(test_set, batch_size=16, shuffle=False)

classes = dataset.classes
print('클래스:', classes)
print(f'train: {len(train_set)}장 / test: {len(test_set)}장')

## 3.모델링

### (1) 필요 함수 생성

* 학습을 위한 함수
* 평가를 위한 함수

Feature Extraction과 Fine-tuning을 같은 방식으로 학습하기 위해 함수로 분리합니다.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()                          # 학습 모드 (Dropout/BN 활성)
    correct, total = 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()              # 이전 step 의 gradient 초기화
        out = model(X)                     # forward
        loss = criterion(out, y)           # 손실 계산
        loss.backward()                    # backward — gradient 구하기
        optimizer.step()                   # 가중치 갱신
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total


def evaluate(model, loader):
    model.eval()                           # 평가 모드 (Dropout/BN 비활성)
    correct, total = 0, 0
    with torch.no_grad():                  # 평가 땐 gradient 안 구함
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / total


print('학습/평가 함수 정의 완료')

### (2) Feature Extraction - 마지막 층만 학습

1. 모든 사전학습 층을 동결합니다.
2. 마지막 `fc`만 4 클래스용 새 Linear 층으로 교체합니다.
3. 옵티마이저에는 `fc` 파라미터만 전달합니다.

In [ ]:
model_fe =




in_features =
model_fe.fc =
model_fe = model_fe.to(device)

# 학습 대상 파라미터 비율 확인 — 굉장히 작음
trainable = sum(p.numel() for p in model_fe.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_fe.parameters())
print(f'학습 파라미터: {trainable:,} / 전체: {total:,} ({trainable/total*100:.2f}%)')

In [ ]:

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_fe.fc.parameters(), lr=1e-3)

epochs =
fe_acc = []
for ep in range(epochs):
    tr = train_one_epoch(model_fe, train_loader, criterion, optimizer)
    te = evaluate(model_fe, test_loader)
    fe_acc.append(te)
    print(f'[FE ep{ep+1}] train_acc={tr:.3f} test_acc={te:.3f}')

### (3) Fine-tuning - 전체 재학습

- 모든 층을 학습 가능 상태로 둡니다.
- 마지막 `fc`만 4 클래스용으로 교체합니다.
- 사전학습 지식이 크게 깨지지 않도록 작은 학습률을 사용합니다.

In [ ]:
model_ft =
model_ft.fc =
model_ft = model_ft.to(device)

optimizer =

ft_acc = []
for ep in range(epochs):
    tr = train_one_epoch(model_ft, train_loader, criterion, optimizer)
    te = evaluate(model_ft, test_loader)
    ft_acc.append(te)
    print(f'[FT ep{ep+1}] train_acc={tr:.3f} test_acc={te:.3f}')

### (4) 전이학습 방식 비교

In [ ]:
# 두 방식 test 정확도 비교
epochs_x = range(1, epochs + 1)
plt.figure(figsize=(8, 5))
plt.plot(epochs_x, fe_acc, 'o-', label='Feature Extraction')
plt.plot(epochs_x, ft_acc, 's-', label='Fine-tuning')
plt.xlabel('Epoch')
plt.ylabel('Test Accuracy')
plt.title('전이학습 방식 비교 — 연예인 4명 분류')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f'FE 최종: {fe_acc[-1]:.3f}')
print(f'FT 최종: {ft_acc[-1]:.3f}')

## 4.결과 해석

직접 만든 작은 CNN과 전이학습 모델의 결과를 비교합니다.

| 방식 | 모델 | epoch | 보통 test 정확도 |
|---|---|---|---|
| 작은 CNN | 직접 만든 Conv 3블록 | 30 | 50~70% |
| Feature Extraction | ResNet18, `fc`만 학습 | 5 | 80~90%+ |
| Fine-tuning | ResNet18 전체 재학습 | 5 | 80~95%+ |

데이터가 매번 새로 다운로드되므로 정확한 숫자는 환경마다 달라질 수 있습니다.

**왜 차이가 나는가**
- 작은 CNN은 적은 이미지로 모든 특징을 처음부터 배워야 합니다.
- ResNet18은 ImageNet에서 일반적인 시각 특징을 이미 학습했습니다.
- 우리는 그 위에 4명을 구분하는 마지막 층만 추가로 학습하거나, 전체를 조금만 조정합니다.

## 정리

내 수집 이미지로 전이학습 분류기를 만드는 흐름:

1. 폴더 구조 만들기: `data/<클래스명>/이미지.jpg`
2. `transform` 정의: Resize + ToTensor + Normalize
3. `ImageFolder`로 데이터 로드
4. `DataLoader`로 배치 구성
5. 사전학습 모델을 불러와 마지막 `fc` 교체
6. **Feature Extraction** 먼저 시도
7. 성능이 부족하면 **Fine-tuning** 시도

- Custom Dataset은 `__init__`, `__len__`, `__getitem__` 세 메서드를 구현합니다.
- ImageFolder는 폴더 구조만 맞추면 한 줄로 데이터를 불러옵니다.
- 사전학습 모델을 쓸 때는 ImageNet 정규화 값 `MEAN`, `STD`를 그대로 사용합니다.

## 연습문제

1. `epochs`를 늘려 두 방식의 최종 정확도 차이를 확인해봅니다.
2. 본인 사진을 한 장 넣어 4명 중 누구로 분류되는지 시험해봅니다.